# Bài 5 v4 · Hồi quy Logistic
PGS.TS. Nguyễn Văn Hậu. Chạy tuần tự 10 ô mã. Toy được thiết kế để tính tay; Olist là dữ liệu thật. Model Olist dùng newton-cg, toy dùng GD.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"source/logistic_lab.py").exists(): raise RuntimeError("Mở notebook ở thư mục Bài 5 đã giải nén")
if (ROOT/".build/pydeps").exists():sys.path.insert(0,str(ROOT/".build/pydeps"))
sys.path.insert(0,str(ROOT/"source"))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.special import expit
from logistic_lab import gradient_descent,gradient,loss,run
from prepare_olist import prepare
plt.rcParams.update({"figure.figsize":(10,5),"axes.grid":True})
OUT=ROOT/"results/lab_figures";OUT.mkdir(parents=True,exist_ok=True)

## 1. Hai vòng đầy đủ
x toy không là km âm và chưa chuẩn hóa mean=0. Thứ tự tham số là (w,b).

In [2]:
x=np.array([-1.,0.,1.,2.]);y=np.array([0.,0.,1.,1.]);h=gradient_descent(x,y,steps=20)
print(pd.DataFrame([{k:r[k] for k in ["step","w","b","loss"]} for r in h[:3]]).to_string(index=False))
print("Gradient cũ",h[0]["gradient"],h[1]["gradient"])
assert np.allclose([h[2]["w"],h[2]["b"]],[.8232410450840703,-.057764644657501224])

 step        w         b     loss
    0 0.000000  0.000000 0.693147
    1 0.500000  0.000000 0.488641
    2 0.823241 -0.057765 0.394775
Gradient cũ [-0.5, 0.0] [-0.32324104508407026, 0.057764644657501224]


In [3]:
xx=np.linspace(-2,3,200);fig,ax=plt.subplots();ax.scatter(x,y,label="Labels (designed)",color="#bc761a")
for r in h[:3]:ax.plot(xx,expit(r["b"]+r["w"]*xx),label=f"t={r[chr(115)+chr(116)+chr(101)+chr(112)]}")
ax.set(xlabel="toy x",ylabel="p",ylim=(-.05,1.05));ax.legend();fig.tight_layout();fig.savefig(OUT/"toy_probabilities.png",dpi=240);plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_11860\980746863.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel="toy x",ylabel="p",ylim=(-.05,1.05));ax.legend();fig.tight_layout();fig.savefig(OUT/"toy_probabilities.png",dpi=240);plt.show()


In [4]:
for th in [(0.,0.),(.5,0.),(.7,-.2)]:
    th=np.array(th)
    for lam in [0.,.1]:
        fd=np.array([(loss(th+np.eye(2)[j]*1e-5,x,y,lam)-loss(th-np.eye(2)[j]*1e-5,x,y,lam))/2e-5 for j in range(2)])
        assert np.allclose(fd,gradient(th,x,y,lam),atol=1e-9)
        print(th,lam,fd)

[0. 0.] 0.0 [-5.00000000e-01  5.55111512e-12]
[0. 0.] 0.1 [-5.00000000e-01  5.55111512e-12]
[0.5 0. ] 0.0 [-0.32324105  0.05776464]
[0.5 0. ] 0.1 [-0.27324105  0.05776464]
[ 0.7 -0.2] 0.0 [-0.2823854   0.03255015]
[ 0.7 -0.2] 0.1 [-0.2123854   0.03255015]


## 2. Bước học và biên
Các đường dùng cùng dữ liệu và khởi tạo. Thay ngưỡng dịch biên, không sửa xác suất.

In [5]:
fig,ax=plt.subplots()
for eta in [.1,1.,10.,100.]:
    hs=gradient_descent(x,y,eta=eta,steps=15);ax.plot([r["step"] for r in hs],[r["loss"] for r in hs],marker="o",label=f"eta={eta}")
ax.set(xlabel="step",ylabel="BCE");ax.legend();fig.tight_layout();fig.savefig(OUT/"learning_rates.png",dpi=240);plt.show()
fig,ax=plt.subplots();u1=np.linspace(-3,3,200)
for tau in [.5,.2]:ax.plot(u1,(.5+np.log(tau/(1-tau))-u1)/.8,label=f"p=tau={tau}")
ax.set(xlabel="toy u1",ylabel="toy u2",xlim=(-3,3),ylim=(-4,4),title="Same logit: z=-0.5+u1+0.8*u2");ax.legend();fig.tight_layout();fig.savefig(OUT/"boundaries.png",dpi=240);plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_11860\500423118.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel="step",ylabel="BCE");ax.legend();fig.tight_layout();fig.savefig(OUT/"learning_rates.png",dpi=240);plt.show()


C:\Users\nguye\AppData\Local\Temp\ipykernel_11860\500423118.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel="toy u1",ylabel="toy u2",xlim=(-3,3),ylim=(-4,4),title="Same logit: z=-0.5+u1+0.8*u2");ax.legend();fig.tight_layout();fig.savefig(OUT/"boundaries.png",dpi=240);plt.show()


## 3. Dữ liệu thật và tái lập
Dựng lại từ ZIP đi kèm rồi fit chỉ train. Không điều chỉnh cấu hình theo test. Test đã xem ở phiên bản trước.

In [6]:
provenance=prepare();audit=run()
assert audit["status"]=="pass"
print("Checks",audit["checks_count"],"weight fixes",provenance["weight_correction_count"])
print("Split",audit["split_counts"])
print("One ID",audit["one_order"])

{"status": "pass", "raw_orders": 99441, "eligible_completed_orders": 96394, "split_counts": {"test": 40266, "train": 29031, "validation": 10389}, "positive_counts": {"test": 3478, "train": 941, "validation": 634}, "weight_correction_count": 16, "weight_corrections_by_split": {"train": 16}, "missing_by_split": {"test": 0, "train": 16, "validation": 0}, "other_columns_preserved": true, "weight_rule": "Total weight is unknown if any item weight is unknown; do not replace unknown with zero.", "distance_recipe": "Median coordinate per postal prefix; haversine radius6371.0088km; mean item distance per order.", "label": "date(delivered)>date(estimated); same calendar day is on time.", "sha256_raw": "967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784", "sha256_prepared": "bdabb4fc0f6c36102aebb0691cc539a7ed6b698084917ec330429bd63ce1d4b1", "limitations": ["Completed-only eligible cohort; train/validation labels complete before cutoff.", "Static product/geographic metadata: historic

{"status": "pass", "checks_count": 33, "coef": [0.48352504620606046, -0.47586647904750623, 0.04484514033269559, 0.007393888091754746, -0.03893708987401564, 0.09157491560069252], "intercept": -3.4868216612313874, "test_selected": {"n": 40266, "tn": 36429, "fp": 359, "fn": 3317, "tp": 161, "accuracy": 0.9087070977996324, "precision": 0.3096153846153846, "precision_defined": true, "recall": 0.04629097182288672, "f1": 0.08054027013506754, "alerts": 520, "cost": 16944, "log_loss": 0.3135988054638768, "brier": 0.08045026540086317, "ap": 0.15923670168606474, "auc": 0.6566554223653325}}
Checks 33 weight fixes 16
Split {'train': 29031, 'validation': 10389, 'test': 40266}
One ID {'order_id': 'fc9bc4bac939adaa600ad59dee9a106f', 'late': 0, 'raw': {'distance_km': 100.0280021542138, 'promised_days': 7.197800925925926, 'price': 59.9, 'freight_value': 11.73, 'n_items': 1.0, 'weight_g': 7000.0}, 'filled': [100.0280021542138, 7.197800925925926, 59.9, 11.73, 1.0, 7000.0], 'standardized': [-0.873674110787

In [7]:
scan=pd.DataFrame(audit["validation_scan"])
print(scan[["policy","threshold","cost","alerts","fn"]].to_string(index=False))
print("Selected",audit["policy"]["selected"])
fig,ax=plt.subplots();ax.plot(scan.threshold,scan.cost,marker="o",label="FP+5FN");ax.axhline(scan.loc[scan.policy=="no_alerts","cost"].iloc[0],color="gray",ls="--",label="No alerts");ax.set(xlabel="threshold (validation)",ylabel="hypothetical K");ax.legend();fig.tight_layout();fig.savefig(OUT/"validation_policy.png",dpi=240);plt.show()

   policy  threshold  cost  alerts  fn
threshold       0.02  8670    9268   6
threshold       0.03  5268    4936 161
threshold       0.05  3374     804 534
threshold       0.10  3109      95 608
threshold       0.15  3167       9 632
threshold       0.20  3173       3 634
threshold       0.30  3173       3 634
threshold       0.50  3172       2 634
threshold       0.80  3172       2 634
no_alerts        NaN  3170       0 634
Selected 0.1


C:\Users\nguye\AppData\Local\Temp\ipykernel_11860\3933106192.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.plot(scan.threshold,scan.cost,marker="o",label="FP+5FN");ax.axhline(scan.loc[scan.policy=="no_alerts","cost"].iloc[0],color="gray",ls="--",label="No alerts");ax.set(xlabel="threshold (validation)",ylabel="hypothetical K");ax.legend();fig.tight_layout();fig.savefig(OUT/"validation_policy.png",dpi=240);plt.show()


In [8]:
pred=pd.read_csv(ROOT/"data/prepared/test_predictions.csv");print(audit["test_selected"])
fig,ax=plt.subplots();bins=np.linspace(0,.5,26)
for label,color in [(0,"#245a84"),(1,"#bc761a")]:
    s=pred.loc[pred.late==label,"score"].to_numpy();ax.hist(s,bins=bins,weights=np.ones(len(s))/len(s),histtype="step",linewidth=2,label=f"y={label}; n={len(s)}",color=color)
ax.axvline(.1,color="#167a72",ls="--",label="tau=.1");ax.set(xlabel="test p",ylabel="probability mass within each class");ax.legend();fig.tight_layout();fig.savefig(OUT/"test_score_histogram.png",dpi=240);plt.show()

{'n': 40266, 'tn': 36429, 'fp': 359, 'fn': 3317, 'tp': 161, 'accuracy': 0.9087070977996324, 'precision': 0.3096153846153846, 'precision_defined': True, 'recall': 0.04629097182288672, 'f1': 0.08054027013506754, 'alerts': 520, 'cost': 16944, 'log_loss': 0.3135988054638768, 'brier': 0.08045026540086317, 'ap': 0.15923670168606474, 'auc': 0.6566554223653325}


C:\Users\nguye\AppData\Local\Temp\ipykernel_11860\1948725993.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.axvline(.1,color="#167a72",ls="--",label="tau=.1");ax.set(xlabel="test p",ylabel="probability mass within each class");ax.legend();fig.tight_layout();fig.savefig(OUT/"test_score_histogram.png",dpi=240);plt.show()


## 4. Xác suất và kết luận
BCE/Brier là scoring rules tổng hợp. Reliability plot cần đọc số mẫu trong bin; không tự chứng minh calibration tương lai.

In [9]:
r=pd.DataFrame(audit["reliability"]);print(r.to_string(index=False))
fig,ax=plt.subplots();ax.plot([0,.35],[0,.35],color="gray",ls="--",label="identity");ax.plot(r.mean_p,r.late_rate,marker="o",label="historical bins");ax.set(xlabel="mean predicted p",ylabel="observed late fraction");ax.legend();fig.tight_layout();fig.savefig(OUT/"reliability.png",dpi=240);plt.show()
print("Logistic",audit["test_selected"]["log_loss"],audit["test_selected"]["brier"])
print("Baseline",audit["baseline"]["log_loss"],audit["baseline"]["brier"])

 lower  upper     n   mean_p  late_rate
  0.00   0.02  7016 0.014782   0.031499
  0.02   0.04 25344 0.029540   0.081913
  0.04   0.06  5636 0.046413   0.118524
  0.06   0.08  1196 0.068599   0.186455
  0.08   0.10   554 0.088486   0.232852
  0.10   0.15   472 0.116971   0.296610
  0.15   0.20    40 0.165878   0.475000
  0.20   0.30     5 0.233583   0.200000
  0.30   1.00     3 0.319994   0.333333


Logistic 0.3135988054638768 0.08045026540086317
Baseline 0.32630163060279926 0.08182675237443353


C:\Users\nguye\AppData\Local\Temp\ipykernel_11860\1520848593.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.plot([0,.35],[0,.35],color="gray",ls="--",label="identity");ax.plot(r.mean_p,r.late_rate,marker="o",label="historical bins");ax.set(xlabel="mean predicted p",ylabel="observed late fraction");ax.legend();fig.tight_layout();fig.savefig(OUT/"reliability.png",dpi=240);plt.show()


In [10]:
from science_audit import run as independent_audit
report=independent_audit();assert report["status"]=="pass"
print("Independent checks",report["checks_count"])
p=expit(np.log(3));print("Exit",p,p/(1-p),-np.log(p),int(p>=.8),p-1)
assert np.isclose(p,.75)
print("Conclusion: ranking improves on this historical cohort; recall at tau=.1 remains low. Costs are hypothetical; a new time period is needed for deployment evidence.")

Independent audit: 75 checks passed.
Independent checks 75
Exit 0.75 3.0 0.2876820724517809 0 -0.25
Conclusion: ranking improves on this historical cohort; recall at tau=.1 remains low. Costs are hypothetical; a new time period is needed for deployment evidence.
